# 01 — ACFA Dataset Exploration & Preprocessing Pipeline Verification

This notebook covers **Milestone 1 & Milestone 2** verification for the ACFA (Acoustic Fault Analyzer) project:
1. Dataset summary and file counts across 4 machine categories (`fan`, `pump`, `slider`/Slide Rail, `valve`).
2. Audio signal properties inspection (sampling rate, duration, channel downmixing, amplitude normalization).
3. Preprocessing pipeline walkthrough: `.wav` → Load → Mix Channels → Normalize → Segment → Mel-Spectrogram.
4. Normal vs. Anomaly waveform and Mel-Spectrogram visualization.
5. Dataset split verification (`train.csv`, `validation.csv`, `test.csv`).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to python path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils import load_wav_file, resolve_filepath, CLASSES
from src.preprocessing import process_file, mix_channels, normalize_amplitude, compute_mel_spectrogram
from src.models import load_acfa_model

print("Project Root:", project_root)
print("Standardized Milestone Classes:", CLASSES)

## 1. Dataset Split Summary & Verification

In [ ]:
splits_dir = os.path.join(project_root, "results", "splits")
train_df = pd.read_csv(os.path.join(splits_dir, "train.csv"))
val_df = pd.read_csv(os.path.join(splits_dir, "validation.csv"))
test_df = pd.read_csv(os.path.join(splits_dir, "test.csv"))

print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print("Total dataset entries:", len(train_df) + len(val_df) + len(test_df))

# Check overlap
train_set = set(train_df["filepath"])
val_set = set(val_df["filepath"])
test_set = set(test_df["filepath"])
print("Train & Val overlap:", len(train_set.intersection(val_set)))
print("Train & Test overlap:", len(train_set.intersection(test_set)))
print("Val & Test overlap:", len(val_set.intersection(test_set)))

## 2. Preprocessing Pipeline Walkthrough
Pipeline steps: `.wav` → Load → 8-to-1 Channel Mix → Peak Normalization → Segment → Mel-Spectrogram

In [ ]:
sample_file = os.path.join(project_root, "dataset", "MIMII", "fan", "normal", "00000000.wav")
sr, raw_audio = load_wav_file(sample_file)
print(f"Raw Audio Shape: {raw_audio.shape} (8 channels, 160k samples)")

mono_audio = mix_channels(raw_audio, mode="mean")
print(f"Mono Mixed Audio Shape: {mono_audio.shape}")

norm_audio = normalize_amplitude(mono_audio, target_peak=1.0)
print(f"Normalized Peak Amplitude: {np.max(np.abs(norm_audio)):.4f}")

spectrograms = process_file(sample_file)
print(f"Generated Mel-Spectrogram Segments: {len(spectrograms)}")
print(f"Mel-Spectrogram Shape per segment: {spectrograms[0].shape}")

## 3. Model Architecture & Checkpoint Compatibility

In [ ]:
ckpt_path = os.path.join(project_root, "results", "models", "acfa_cnn_best.pth")
model = load_acfa_model(ckpt_path, device="cpu")
print("ACFACNN Model successfully loaded!")
print(model)